# Agents

Một agent là một model gọi các tool theo một vòng lặp cho đến khi hoàn thành một nhiệm vụ nhất định.

<p align="center">
  <img 
    src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/core_agent_loop.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=4b4cbb497b6273758a565de1bc90ece0" 
    alt="Sơ đồ vòng lặp agent cơ bản" 
    style="height: 450px; width: auto; border-radius: 8px;"
  />
</p>

Một harness là toàn bộ những gì bao quanh vòng lặp đó: prompt, các tool, và bất kỳ middleware nào định hình hành vi của model.

**Agent = Model + Harness**. Nhiệm vụ của harness là: cung cấp cho model đúng context vào đúng thời điểm cho nhiệm vụ đã cho.

`create_agent` là một harness có khả năng cấu hình cao. Dựa trên nền tảng đó, bạn có thể cấu hình các thành phần cơ bản trực tiếp bằng các tham số `model=`, `tools=`, và `system_prompt=`.

## Các thành phần cốt lõi

<p align="center">
  <img 
    src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/agent_model_harness.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=5ac6a7e0343af7cb5ba3ca632e2224af" 
    alt="Sơ đồ các thành phần model và harness của agent" 
    style="height: 450px; width: auto; border-radius: 8px;"
  />
</p>

### Model

Truyền vào một chuỗi định danh model (`"provider:model"`) hoặc một instance model đã được khởi tạo để chọn model cho agent của bạn.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model="google_genai:gemini-3.6-flash")

### Tools

Để cung cấp tool cho agent, hãy truyền vào bất kỳ Python callable, LangChain tool, hoặc tool dict nào.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Tìm kiếm thông tin."""
    return f"Kết quả cho: {query}"


agent = create_agent(model="google_genai:gemini-3.6-flash", tools=[search])

### System prompt

Định hình cách agent tiếp cận các nhiệm vụ. Tham số system prompt nhận vào một chuỗi hoặc `SystemMessage`.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    system_prompt="Bạn là một trợ lý hữu ích. Hãy trả lời ngắn gọn và chính xác.",
)

### Structured output

Trả về một schema đã được xác thực từ agent bằng cách sử dụng `response_format=`.

In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent


class Answer(BaseModel):
    summary: str
    confidence: float


agent = create_agent(model="google_genai:gemini-3.6-flash", response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Tóm tắt các xu hướng AI"}]})
result["structured_response"]

Answer(summary='Các xu hướng AI hiện nay bao gồm sự bùng nổ của mô hình ngôn ngữ lớn (LLM) và AI tạo sinh, tích hợp AI vào thiết bị cá nhân (AI PC/Smartphone), phát triển AI đa phương thức (multimodal), và sự chú trọng vào tính an toàn, đạo đức cũng như hiệu quả năng lượng của mô hình.', confidence=0.95)

### Agent state

Mỗi agent quản lý execution context của mình thông qua `AgentState`, một typed dictionary lưu giữ lịch sử hội thoại hiện tại và bất kỳ trường tùy chỉnh nào mà tool và middleware của bạn cần.

Trường được tích hợp sẵn là:

| Trường     | Kiểu dữ liệu        | Mô tả                                                                                                       |
| ---------- | ------------------- | ------------------------------------------------------------------------------------------------------------ |
| `messages` | `list[BaseMessage]` | Toàn bộ lịch sử hội thoại của thread hiện tại. Chỉ được nối thêm: các message mới được thêm vào, không bao giờ bị thay thế. |

`AgentState` cũng là kiểu dữ liệu cho mọi node-style middleware hook (`before_model`, `after_model`, và tương tự). Các hook nhận state hiện tại và có thể trả về một dict chứa các cập nhật để hợp nhất trở lại vào state đó.

Để thêm các trường tùy chỉnh (ví dụ: `user_id` hoặc một counter), hãy subclass `AgentState` và truyền subclass đó vào `create_agent` qua `state_schema=`:

In [ ]:
from langchain.agents import AgentState, create_agent


class MyState(AgentState):
    user_id: str
    call_count: int


agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    state_schema=MyState,
)

## Gọi thực thi

Bạn có thể gọi thực thi một agent bằng một message. Về bản chất, việc này sẽ truyền một cập nhật đến `State` của agent. Mọi agent đều bao gồm một chuỗi các message trong state của mình; để gọi thực thi agent, hãy truyền vào một message mới cùng với một `thread_id` để agent có thể lưu giữ và tiếp tục lịch sử hội thoại:

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Thời tiết ở San Francisco thế nào?"}]},
    config=config,
)

# Một lượt tiếp theo trong cùng hội thoại: dùng lại cùng thread_id để giữ lịch sử
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Còn ngày mai thì sao?"}]},
    config=config,
)

Nếu bạn cũng cần truyền cấu hình theo từng lần chạy - chẳng hạn user ID, API key, hoặc feature flag - cho tool và middleware, hãy truyền dữ liệu đó dưới dạng `context` cùng với `config`. Định nghĩa cấu trúc của dữ liệu đó bằng `context_schema` và truy cập nó qua `runtime.context`:

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver


@dataclass
class Context:
    user_id: str


agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    context_schema=Context,
    checkpointer=InMemorySaver(),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Thời tiết ở San Francisco thế nào?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
    context=Context(user_id="user-123"),
)

`thread_id` xác định phạm vi của *hội thoại* (lịch sử message, checkpoint), còn `context` mang dữ liệu *theo từng lần chạy* mà tool và middleware của bạn đọc tại thời điểm gọi thực thi. Cả hai thường được truyền cùng nhau.

## Streaming

`invoke` trả về phản hồi cuối cùng khi kết thúc một lần chạy. Nếu một agent thực thi nhiều lệnh gọi tool, người dùng thường cần cập nhật về tiến trình trước khi hoàn tất. Hãy sử dụng streaming để hiển thị các message trung gian và hoạt động của tool ngay khi chúng diễn ra.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage


agent = create_agent(model="google_genai:gemini-3.6-flash")

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Tìm kiếm tin tức về AI và tóm tắt các kết quả"}]},
    version="v3",
)
for snapshot in stream.values:
    # Mỗi snapshot chứa toàn bộ state tại thời điểm đó
    latest_message = snapshot["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Đang gọi tool: {[tc['name'] for tc in latest_message.tool_calls]}")

User: Tìm kiếm tin tức về AI và tóm tắt các kết quả
Agent: [{'type': 'text', 'text': 'Dưới đây là tổng hợp các tin tức nổi bật và quan trọng nhất về trí tuệ nhân tạo (AI) trong những ngày gần đây:\n\n### 1. OpenAI ra mắt tính năng "Advanced Voice Mode" (Chế độ giọng nói nâng cao)\n*   **Thông tin:** OpenAI đã bắt đầu triển khai tính năng giọng nói nâng cao cho người dùng ChatGPT Plus. \n*   **Điểm nhấn:** Tính năng này cho phép trò chuyện với AI theo thời gian thực với độ trễ cực thấp, giọng nói tự nhiên, có cảm xúc và khả năng ngắt lời hoặc thay đổi giọng điệu dựa trên yêu cầu của người dùng. Đây được coi là bước tiến lớn trong việc tương tác người - máy.\n\n### 2. Sự bùng nổ của các mô hình AI nhỏ (Small Language Models - SLMs)\n*   **Thông tin:** Thay vì chỉ tập trung vào các mô hình khổng lồ, các tập đoàn như Google (Gemma), Microsoft (Phi-3) và Apple đang chuyển hướng mạnh mẽ sang các mô hình AI nhỏ, hiệu quả cao.\n*   **Điểm nhấn:** Các mô hình này có thể chạy trực tiếp trên thiế

## Cấu hình harness

`create_agent` có khả năng mở rộng cao. Middleware là thành phần nguyên thủy để tùy chỉnh: mỗi phần xử lý một mối quan tâm riêng, gắn vào vòng lặp của agent tại đúng thời điểm, và kết hợp tự do với bất kỳ thành phần nào khác. Chỉ cần lấy đúng những gì use case của bạn cần và bỏ qua phần còn lại.

Các pattern phổ biến đã được lắp sẵn dưới dạng middleware hạng nhất. Bạn có thể tự xây dựng bất kỳ thứ gì khác dưới dạng middleware tùy chỉnh.

<p align="center">
  <img 
    src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/agent_harness_capabilities.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=0ff671d72badd0844826660dfcb04391" 
    alt="Các khả năng của agent harness theo từng hạng mục" 
    style="height: 300px; width: auto; border-radius: 8px;"
  />
</p>

Khi agent đảm nhận các công việc phức tạp, chúng cần được hỗ trợ ở một số mảng chính. Hệ sinh thái middleware cung cấp:
- Môi trường thực thi: Tool, hệ thống file, sandbox, và thực thi mã
- Quản lý context: Summarization, memory, skill, và prompt caching
- Lập kế hoạch và phân bổ công việc: Danh sách todo và subagent cho công việc song song, độc lập
- Khả năng chịu lỗi: Retry, fallback, và giới hạn số lượt gọi
- Guardrails: Phát hiện PII và kiểm soát nội dung
- Steering: Con người phê duyệt (human-in-the-loop) trước các hành động có tác động lớn

> `create_deep_agent` lắp sẵn stack này cho các nhiệm vụ coding và research chạy dài (bao gồm sẵn hệ thống file, summarization, subagent, và prompt caching theo mặc định).

### Môi trường thực thi

Agent đặc biệt hữu ích khi chúng có thể thực hiện hành động thay vì chỉ tạo ra văn bản. Môi trường thực thi cung cấp cho agent một không gian làm việc: các tool nó có thể gọi, một hệ thống file để đọc và viết file qua nhiều lượt, và khả năng thực thi mã để chạy script hoặc lệnh shell.

In [ ]:
from langchain.agents import create_agent
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[search],
    middleware=[FilesystemMiddleware(backend=StateBackend())],
)

### Quản lý context

Mỗi lệnh gọi model có một context window cố định. Khi agent chạy, context window đó dần được lấp đầy bởi lịch sử tích lũy, kết quả từ tool, và các bước trung gian. Summarization nén lịch sử lại trước khi xảy ra tràn; memory nạp các chỉ dẫn lâu dài khi khởi động để kiến thức được duy trì qua các session; skill cung cấp kiến thức chuyên ngành theo yêu cầu thay vì nạp hết mọi thứ ngay từ đầu.

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware, MemoryMiddleware, SkillsMiddleware, SummarizationMiddleware

backend = StateBackend()
model = "google_genai:gemini-3.6-flash"

agent = create_agent(
    model=model,
    tools=[search],
    middleware=[
        FilesystemMiddleware(backend=backend),
        SummarizationMiddleware(model=model, backend=backend),
        MemoryMiddleware(backend=backend, sources=["./AGENTS.md"]),
        SkillsMiddleware(backend=backend, sources=["./skills/"]),
    ],
)

### Lập kế hoạch và phân bổ công việc

Các nhiệm vụ phức tạp thường vượt quá khả năng xử lý của một context window. Việc phân bổ công việc cho phép agent chính chia nhỏ công việc thành nhiều phần, giao chúng cho các subagent - mỗi subagent chạy trong context riêng biệt của mình - và giữ cho agent chính tập trung vào việc điều phối thay vì thực thi. Công việc có thể chạy song song; context của agent chính vẫn được giữ gọn gàng.

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về một bản tóm tắt ngắn."""
    return f"Kết quả tìm kiếm cho: {query}"


backend = StateBackend()

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[search],
    middleware=[
        FilesystemMiddleware(backend=backend),
        TodoListMiddleware(),
        SubAgentMiddleware(
            backend=backend,
            subagents=[
                {
                    "name": "researcher",
                    "description": "Tìm kiếm và trả về một bản tóm tắt có cấu trúc.",
                    "system_prompt": "Sử dụng tool search để nghiên cứu câu hỏi và tóm tắt các điểm chính.",
                    "tools": [search],
                    "model": "anthropic:claude-sonnet-4-6",
                    "middleware": [],
                }
            ],
        ),
    ],
)

### Đặt tên cho agent

Tùy chọn sử dụng một định danh cho agent. Điều này đặc biệt hữu ích khi nhúng agent như một subgraph trong các hệ thống multi-agent.

In [ ]:
agent = create_agent(model="google_genai:gemini-3.6-flash", name="research_assistant")

### Khả năng chịu lỗi

Các agent trong môi trường production gặp phải những lỗi hiếm khi xuất hiện trong quá trình phát triển: rate limit, model timeout, lỗi API tạm thời. Middleware chịu lỗi (fault tolerance) xử lý những vấn đề này ở tầng hạ tầng để tool và logic nghiệp vụ của bạn không cần try/catch xung quanh mỗi lệnh gọi.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về một bản tóm tắt ngắn."""
    return f"Kết quả tìm kiếm cho: {query}"


agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[search],
    middleware=[
        ModelRetryMiddleware(max_retries=3),
        ToolRetryMiddleware(max_retries=2),
    ],
)

### Guardrails

Một số chính sách không thể chỉ nằm trong prompt - chúng cần được thực thi một cách xác định, bất kể model làm gì. Guardrails can thiệp vào dữ liệu khi nó chảy qua vòng lặp của agent, áp dụng các quy tắc tuân thủ hoặc chính sách nội dung trước khi kết quả từ tool đến được context của model.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về một bản tóm tắt ngắn."""
    return f"Kết quả tìm kiếm cho: {query}"


agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[search],
    middleware=[PIIMiddleware("email")],
)

### Steering

Quyền tự chủ hoàn toàn không phải lúc nào cũng phù hợp. Steering cho phép bạn đặt con người vào các điểm quyết định cụ thể - trước các thao tác viết có tính phá hủy, các lệnh gọi API tốn kém, hoặc bất kỳ điều gì cần đến sự đánh giá - mà không cần cấu trúc lại agent của bạn. Agent sẽ tạm dừng và chờ; con người phê duyệt, chỉnh sửa, hoặc từ chối; sau đó việc thực thi tiếp tục.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về một bản tóm tắt ngắn."""
    return f"Kết quả tìm kiếm cho: {query}"


agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[search],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"write_file": True})],
)